# 02 · Measure what each lesson teaches

**Goal:** build one source student's outcome table from independent
adaptation branches, all starting at the same checkpoint.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](README.md)
· [HAIC setup and launch commands](../../slurm/synthetic-training/README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

## 1. Select a source student

Set `ST_STUDENT_ID` before starting this notebook. The Slurm launcher
creates one array task per configured train/validation student. Held
architectures are excluded from this stage.

The original estimator is $M_0$. Record its target predictions and
labeled diagnostic errors, apply the common probe, and obtain $M_p$.
Record predictions again on exactly the same frames. Coordinate changes
measure the response to training, not whether the change is correct.

In [ ]:
STUDENT_ID = os.environ.get("ST_STUDENT_ID")
if not STUDENT_ID:
    raise ValueError("Set ST_STUDENT_ID to one configured train/validation student.")
student = cfg.student(STUDENT_ID)
if student["role"] == "held":
    raise ValueError("Held students cannot enter source teaching trials.")
display(pd.DataFrame([student]))
print(f"Probe updates: {cfg.probe_steps}; remaining budgets: {cfg.adaptation_steps}")

## 2. Fork each candidate from the same post-probe student

For every declared remaining budget, independently adapt $M_p$ using
replay alone or replay plus one lesson. Do not train the lessons in a
sequence. Give each branch the same optimizer initialization and update
count. Reuse a trained branch across reference settings instead of
repeating the training job for each setting.

The teacher's target is **gain over replay**:

$$\mathrm{gain}(k) = E_{\mathrm{replay}} - E_k.$$

Positive gain means lesson $k$ reduced independent reference error more
than replay. Negative gain means it was worse. Replay has gain zero and
remains an available choice. Full-budget replay from $M_0$ is also
retained to test whether the whole probe-and-lesson procedure is useful.

In [ ]:
started = perf_counter()
trials = workflow.source_trials(cfg, student_id=STUDENT_ID)
show_result(trials)
print(f"Student trials took {(perf_counter() - started) / 3600:.2f} hours.")

## 3. Look for a selection opportunity

Different lessons must produce meaningfully different gains. Different
students or contexts should sometimes favor different choices. Uniform
gains are ordinary augmentation, not evidence for a personalized teacher.

Many domain rows reuse the same adapted checkpoint. They are not
independent training experiments. Count students and independent
interventions alongside the number of rows in the outcome table.

Source trials may use labeled synthetic references to measure every
lesson. Deployment will choose one lesson before accessing real labels.

After all source tasks complete, continue to
[03 · Fit and freeze selectors](03_fit_and_freeze_selectors.ipynb).